## ⚠ Save a copy to your Drive first

**This notebook is fetched fresh from GitHub every time you open the link.** Any edits you make here — settings, code, hyperparameters — **will be LOST when you close the tab** unless you save a copy.

**To keep your edits:**

1. **File → Save a copy in Drive** (top menu)
2. Re-open the saved copy via **File → Open notebook → Recent** or your Google Drive next time

The saved copy is yours to edit; the GitHub link always opens fresh.

# Train a Tabular Classifier — RandomForest (simple)

**Maintained by:** IGNODE  
**Last verified:** May 2026 against scikit-learn 1.5  
**Runtime:** under 1 minute on Colab's free CPU tier

Train a tabular classification model using **RandomForest** — an ensemble of decision trees. Often the most interpretable choice for tabular data; great when your dataset is small (<1000 rows) or when you want to inspect feature importances. This notebook is **linear** — no branching, no AutoML — read it end-to-end.

If you want to compare algorithms or run AutoML, use `train_tabular_classifier.ipynb` instead.

## Output

`model.onnx` + `class_labels.json` + `feature_columns.json` — drop into IGNODE → ML Factory → Custom Models → + Upload ML Model.

## Quick start

### To try it right now (no setup needed)

1. **Runtime → Run all** at the top of Colab
2. Wait ~30 seconds for dependencies + ~10 seconds for training
3. The last cell automatically downloads `model.onnx` + two sidecar JSONs to your laptop — that's a trained RandomForest classifier on the sample IoT data

### To train on YOUR data

You only need to edit **two values** anywhere in the whole notebook:

| Step | Cell | What to change |
|---|---|---|
| 1 | **Load data** cell (below) | `SAMPLE_DATASET = 'sensor_anomaly_classification'` → `SAMPLE_DATASET = None` |
| 2 | **Settings** cell | `LABEL_COLUMN = 'Anomaly'` → `LABEL_COLUMN = 'your_target_column'` |

Then **Runtime → Run all** again. The load-data cell will prompt you to upload your CSV; everything downstream adapts automatically.

### What you get at the end

- `model.onnx` — your trained model
- `class_labels.json` — the class names
- `feature_columns.json` — input contract (`{feature_columns, label_columns}`)
- **Bonus** — RandomForest prints **feature importance ranking** so you can see which features the model relies on most

Drop the artifacts into **IGNODE → ML Factory → Custom Models → + Upload ML Model**.

## 1. Install pinned dependencies

In [ ]:
# Install the libraries this notebook needs.
#
# `--quiet` is omitted on purpose — install errors are visible here
# instead of becoming ModuleNotFoundError later.
#
# onnx-ecosystem packages are NOT pinned (these libraries co-evolve; exact
# pins break with each release). scikit-learn IS pinned for reproducibility.
!pip install scikit-learn==1.5.2 \
             onnx skl2onnx onnxconverter-common

# Verify everything imports cleanly. Failed installs fail fast here
# instead of much later in the export cell.
import sklearn
import skl2onnx  # noqa: F401 — used by the export cell
print(f'scikit-learn: {sklearn.__version__}')

## 2. Load data

Ships set to a small IoT sample (`sensor_anomaly_classification.csv`) so the notebook runs end-to-end out of the box. To use your own CSV, set `SAMPLE_DATASET = None` below and update `LABEL_COLUMN` in the Settings cell.

In [ ]:
# ───────── EDIT THIS ─────────
SAMPLE_DATASET = 'sensor_anomaly_classification'   # set to None to upload your own CSV
# ────────────────────────────

import pandas as pd

if SAMPLE_DATASET:
    url = f'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/examples/{SAMPLE_DATASET}.csv'
    df = pd.read_csv(url)
    print(f'Loaded sample {SAMPLE_DATASET!r}: {df.shape[0]} rows x {df.shape[1]} columns')
else:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded.keys()))
    df = pd.read_csv(csv_path)
    print(f'Loaded {csv_path}: {df.shape[0]} rows x {df.shape[1]} columns')

display(df.head())

## 3. Settings

In [ ]:
# ───────── EDIT THESE ─────────
LABEL_COLUMN = 'Anomaly'   # sample default; change when you bring your own CSV

# RandomForest hyperparameters
N_ESTIMATORS = 200        # number of trees
MAX_DEPTH = None          # None = grow until each leaf is pure
MIN_SAMPLES_LEAF = 1      # minimum samples in a leaf — raise to reduce overfitting

TEST_SIZE = 0.2
RANDOM_SEED = 42
# ──────────────────────────────

if LABEL_COLUMN not in df.columns:
    raise ValueError(f"Label column '{LABEL_COLUMN}' not in CSV. Available: {list(df.columns)}")

## 4. Prep the data

In [ ]:
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

def normalize(name):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', name).strip('_')

rename_map = {c: normalize(c) for c in df.columns if c != normalize(c)}
if rename_map:
    print('Renamed columns:')
    for old, new in rename_map.items():
        print(f'  {old!r}  ->  {new!r}')
    df = df.rename(columns=rename_map)
    if LABEL_COLUMN in rename_map:
        LABEL_COLUMN = rename_map[LABEL_COLUMN]

X = df.drop(columns=[LABEL_COLUMN])
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df[LABEL_COLUMN])
class_labels = [str(c) for c in label_encoder.classes_]
feature_columns = list(X.columns)

print(f'Features ({len(feature_columns)}): {feature_columns}')
print(f'Classes  ({len(class_labels)}):  {class_labels}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {X_train.shape[0]} rows. Test: {X_test.shape[0]} rows.')

## 5. Train

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier

t0 = time.time()
model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print(f'Trained in {time.time() - t0:.1f} sec')

## 6. Evaluate

Accuracy, per-class precision / recall / F1, a confusion matrix, plus a **feature-importance ranking** — one of the reasons RandomForest is often picked for interpretability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

y_pred = model.predict(X_test)
print(f'Test-set accuracy: {accuracy_score(y_test, y_pred):.3f}\n')
print(classification_report(y_test, y_pred, target_names=class_labels, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=class_labels).plot(
    ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False
)
ax.set_title('Confusion Matrix (test set)')
plt.tight_layout()
plt.show()

# Feature importances (RandomForest-specific bonus)
importances = model.feature_importances_
order = np.argsort(importances)[::-1]
print('\nFeature importances (most → least useful):')
for idx in order:
    print(f'  {feature_columns[idx]:<30} {importances[idx]:.4f}')

## 7. Export to ONNX

Convert the sklearn RandomForest model to ONNX using `skl2onnx.convert_sklearn`.

In [ ]:
from skl2onnx import convert_sklearn
from onnxconverter_common.data_types import FloatTensorType
import onnx

initial_types = [('input', FloatTensorType([None, len(feature_columns)]))]

onnx_model = None
for opset in (18, 15, 12):
    try:
        onnx_model = convert_sklearn(model, initial_types=initial_types, target_opset=opset)
        print(f'Exported at opset {opset}')
        break
    except Exception as ex:
        print(f'  opset {opset} failed ({type(ex).__name__}); trying lower')
if onnx_model is None:
    raise RuntimeError('All opset attempts failed.')

onnx.save_model(onnx_model, 'model.onnx')
print(f'Saved model.onnx ({len(onnx_model.SerializeToString()) / 1024:.1f} KB)')

## 8. Write sidecar files

In [ ]:
import json

with open('class_labels.json', 'w') as f:
    # Trainer-aligned shape (matches ignode-trainer-tabular sidecars.py):
    # {schema_version, class_labels} — IGNODE re-imports the array via
    # the .class_labels key from the wrapped object.
    json.dump({'schema_version': 1, 'class_labels': class_labels}, f, indent=2)

# Trainer-aligned shape (matches ignode-trainer-tabular sidecars.py):
# {schema_version, feature_columns} + IGNODE-specific label_columns (used
# by the Custom Model Upload wizard to populate the Target Column field).
feature_sidecar = {'schema_version': 1, 'feature_columns': feature_columns, 'label_columns': [LABEL_COLUMN]}
with open('feature_columns.json', 'w') as f:
    json.dump(feature_sidecar, f, indent=2)

print('class_labels.json:')
print(json.dumps(class_labels, indent=2))
print('\nfeature_columns.json:')
print(json.dumps(feature_sidecar, indent=2))

## 9. Download

In [ ]:
import zipfile
from google.colab import files

# Bundle every artifact into one ZIP so the customer only clicks Download once.
ARTIFACTS = ['model.onnx', 'class_labels.json', 'feature_columns.json']

with zipfile.ZipFile('model-artifacts.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ARTIFACTS:
        zf.write(fname)

files.download('model-artifacts.zip')

## 10. Upload to IGNODE

1. **Integrations → ML Factory** in your IGNODE portal
2. Switch to the **Custom Models** tab
3. Click **+ Upload ML Model**
4. Drop your `model.onnx` and fill in the metadata form:
    - **Task Type:** `Classification`
    - **Class Labels:** paste from `class_labels.json`
    - **Feature Columns:** paste from `feature_columns.json` → `feature_columns`
    - **Target Column:** paste from `feature_columns.json` → `label_columns` (single entry)
5. Click **Upload**, then **Open in Playground** to test.

---

## Reusing this notebook for your own data

Two edits switch to your own dataset:

```python
# In the "Load data" cell:
SAMPLE_DATASET = None        # was 'sensor_anomaly_classification'

# In the "Settings" cell:
LABEL_COLUMN = 'YourColumn'  # was 'Anomaly' — whatever you want to predict
```

Everything else adapts automatically.

### Optional tweaks

| Want to change | Edit (Settings cell) |
|---|---|
| Forest size | `N_ESTIMATORS = 500` (more trees = more stable, slower) |
| Limit depth | `MAX_DEPTH = 10` (None = unlimited; lower = less overfitting) |
| Leaf minimum | `MIN_SAMPLES_LEAF = 5` (higher = less overfitting) |
| Train/test ratio | `TEST_SIZE = 0.3` |
| Reproducibility | `RANDOM_SEED = <any int>` |

### When to pick RandomForest over LightGBM / XGBoost

- **Small datasets** (< 1000 rows) — RandomForest is robust where boosting can overfit
- **You want interpretability** — the feature-importance ranking tells you what the model leans on
- **Mixed feature types** — RandomForest handles numeric + categorical (after encoding) without much tuning
- **Reproducibility matters** — RandomForest is very stable across runs with the same `RANDOM_SEED`

### Common errors

| Error | Fix |
|---|---|
| `Label column 'X' not in CSV` | Check the column list printed by the inspect cell |
| Upload rejected: "invalid column names" | Re-export your CSV with the renamed columns shown in the prep cell |
| Upload rejected: "duplicate name" | Pick a different name in the upload wizard, or delete the existing model |
| Overfitting (test accuracy << train accuracy) | Raise `MIN_SAMPLES_LEAF` to 5 or 10, or lower `MAX_DEPTH` to 10-15 |